# Feature Engineering

## Imports and Configuration

In [4]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIRECTORY = PROJECT_ROOT / "data" / "generated"

TELEMETRY_PATH = DATA_DIRECTORY / "synthetic_telemetry.csv"
DEVICES_PATH = DATA_DIRECTORY / "device_registry.csv"
COMMANDS_PATH = DATA_DIRECTORY / "synthetic_commands.csv"

PREDICTION_INTERVAL_MINUTES = 15
PREDICTION_HORIZON_MINUTES = 60

LAG_INTERVALS = [1, 2, 4]
ROLLING_WINDOW_READINGS = 4

RANDOM_SEED = 42

for path in [TELEMETRY_PATH, DEVICES_PATH, COMMANDS_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Required dataset not found: {path}")

print("Feature Engineering configuration Loaded")
print(f"Prediction Horizon: {PREDICTION_HORIZON_MINUTES} minutes")
print(f"Prediction Interval: {PREDICTION_INTERVAL_MINUTES} minutes")

Feature Engineering configuration Loaded
Prediction Horizon: 60 minutes
Prediction Interval: 15 minutes


## Load Datasets

In [5]:
telemetry_df = pd.read_csv(
    TELEMETRY_PATH,
    parse_dates=["timestamp"],
)

devices_df = pd.read_csv(DEVICES_PATH)

commands_df = pd.read_csv(
    COMMANDS_PATH,
    parse_dates=["timestamp"],
)

telemetry_df = telemetry_df.sort_values(
    ["device_id", "timestamp"]
).reset_index(drop=True)

commands_df = commands_df.sort_values(
    ["device_id", "timestamp"]
).reset_index(drop=True)

# Ensure each device has at most one telemetry row per timestamp
if telemetry_df.duplicated(["device_id", "timestamp"]).any():
    raise ValueError("Duplicate device/timestamp telemetry records detected.")

if telemetry_df["timestamp"].isna().any():
    raise ValueError("Telemetry contains missing timestamps.")

if commands_df["timestamp"].isna().any():
    raise ValueError("Command history contains missing timestamps.")

print(f"Telemetry Rows: {len(telemetry_df):,}")
print(f"Registered Devices: {devices_df['device_id'].nunique():,}")
print(f"Recorded Commands: {len(commands_df):,}")

display(telemetry_df.head())

Telemetry Rows: 34,560
Registered Devices: 12
Recorded Commands: 154


,timestamp,device_id,node_id,zone,device_category,temperature_c,humidity_pct,moisture_pct,water_ph,water_ec_ms_cm,water_flow_l_min,device_health,battery_pct,pump_active,fan_active
0,2026-01-01 00:00:00+00:00,SX-ENV-001,NF-001,ZONE-A,Environmental,23.29,71.41,49.59,NaN,NaN,NaN,0.974,100.00,False,False
1,2026-01-01 00:15:00+00:00,SX-ENV-001,NF-001,ZONE-A,Environmental,23.07,67.39,50.87,NaN,NaN,NaN,0.926,100.00,False,False
2,2026-01-01 00:30:00+00:00,SX-ENV-001,NF-001,ZONE-A,Environmental,22.57,64.92,45.83,NaN,NaN,NaN,0.961,100.00,False,False
3,2026-01-01 00:45:00+00:00,SX-ENV-001,NF-001,ZONE-A,Environmental,24.00,72.27,47.24,NaN,NaN,NaN,0.973,98.87,False,False
4,2026-01-01 01:00:00+00:00,SX-ENV-001,NF-001,ZONE-A,Environmental,22.81,67.66,49.91,NaN,NaN,NaN,0.959,100.00,False,False


## Verify Telemetry sampling intervals

In [6]:
telemetry_df["interval_minutes"] = (
    telemetry_df.groupby("device_id")["timestamp"]
    .diff()
    .dt.total_seconds()
    .div(60)
)

sampling_summary = (
    telemetry_df.groupby("device_id")["interval_minutes"]
    .agg(["count", "median", "min", "max"])
    .reset_index()
)

display(sampling_summary)

unexpected_intervals = telemetry_df.loc[
    telemetry_df["interval_minutes"].notna()
    & ~np.isclose(
        telemetry_df["interval_minutes"],
        PREDICTION_INTERVAL_MINUTES,
    )
]

print(f"Unexpected intervals: {len(unexpected_intervals):,}")

telemetry_df = telemetry_df.drop(columns=["interval_minutes"])

,device_id,count,median,min,max
0,SX-ENV-001,2879,15.0,15.0,15.0
1,SX-ENV-002,2879,15.0,15.0,15.0
2,SX-ENV-003,2879,15.0,15.0,15.0
3,SX-ENV-004,2879,15.0,15.0,15.0
4,SX-ENV-005,2879,15.0,15.0,15.0
5,SX-ENV-006,2879,15.0,15.0,15.0
6,SX-ENV-007,2879,15.0,15.0,15.0
7,SX-WTR-001,2879,15.0,15.0,15.0
8,SX-WTR-002,2879,15.0,15.0,15.0
9,SX-WTR-003,2879,15.0,15.0,15.0


Unexpected intervals: 0


## Define feature Columns

In [7]:
# Numeric telemetry measurements used to describe device conditions
sensor_features = [
    "temperature_c",
    "humidity_pct",
    "moisture_pct",
    "water_ph",
    "water_ec_ms_cm",
    "water_flow_l_min",
    "device_health",
    "battery_pct",
]

# Deployment context provides useful information about each device
context_features = [
    "device_category",
    "zone",
]

feature_df = telemetry_df[
    ["timestamp", "device_id"] + context_features + sensor_features
].copy()

feature_df = feature_df.sort_values(
    ["device_id", "timestamp"]
).reset_index(drop=True)

print(f"Initial feature rows: {len(feature_df):,}")

Initial feature rows: 34,560


## Create lag Features

In [8]:
# Capture historical sensor readings for each device
for column in sensor_features:
    for lag in LAG_INTERVALS:
        feature_df[f"{column}_lag_{lag}"] = (
            feature_df.groupby("device_id")[column].shift(lag)
        )

print("Lag features Created")
print(f"Feature columns so far: {len(feature_df.columns)}")

Lag features Created
Feature columns so far: 36


## Create rolling Statistics

In [9]:
# Calculate rolling means and standard deviations
for column in sensor_features:
    grouped = feature_df.groupby("device_id")[column]

    feature_df[f"{column}_rolling_mean"] = grouped.transform(
        lambda values: values.rolling(
            window=ROLLING_WINDOW_READINGS,
            min_periods=2,
        ).mean()
    )

    feature_df[f"{column}_rolling_std"] = grouped.transform(
        lambda values: values.rolling(
            window=ROLLING_WINDOW_READINGS,
            min_periods=2,
        ).std()
    )

print("Rolling statistics Created")
print(f"Total feature columns: {len(feature_df.columns)}")

Rolling statistics Created
Total feature columns: 52


## Create trend Features

In [10]:
for column in sensor_features:
    feature_df[f"{column}_change_30m"] = (
        feature_df[column]
        - feature_df.groupby("device_id")[column].shift(2)
    )

display(feature_df.head())

,timestamp,device_id,device_category,zone,temperature_c,humidity_pct,moisture_pct,water_ph,water_ec_ms_cm,water_flow_l_min,...,battery_pct_rolling_mean,battery_pct_rolling_std,temperature_c_change_30m,humidity_pct_change_30m,moisture_pct_change_30m,water_ph_change_30m,water_ec_ms_cm_change_30m,water_flow_l_min_change_30m,device_health_change_30m,battery_pct_change_30m
0,2026-01-01 00:00:00+00:00,SX-ENV-001,Environmental,ZONE-A,23.29,71.41,49.59,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-01-01 00:15:00+00:00,SX-ENV-001,Environmental,ZONE-A,23.07,67.39,50.87,NaN,NaN,NaN,...,100.0000,0.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-01-01 00:30:00+00:00,SX-ENV-001,Environmental,ZONE-A,22.57,64.92,45.83,NaN,NaN,NaN,...,100.0000,0.000,-0.72,-6.49,-3.76,NaN,NaN,NaN,-0.013,0.00
3,2026-01-01 00:45:00+00:00,SX-ENV-001,Environmental,ZONE-A,24.00,72.27,47.24,NaN,NaN,NaN,...,99.7175,0.565,0.93,4.88,-3.63,NaN,NaN,NaN,0.047,-1.13
4,2026-01-01 01:00:00+00:00,SX-ENV-001,Environmental,ZONE-A,22.81,67.66,49.91,NaN,NaN,NaN,...,99.7175,0.565,0.24,2.74,4.08,NaN,NaN,NaN,-0.002,0.00


## Inspect missing Feature Values

In [11]:
# Missing values at the beginning of each device history are expected
feature_missingness = (
    feature_df.isna()
    .sum()
    .sort_values(ascending=False)
    .rename("missing_count")
    .to_frame()
)

feature_missingness["missing_percentage"] = (
    feature_missingness["missing_count"]
    / len(feature_df)
    * 100
).round(2)

display(feature_missingness.head(15))

,missing_count,missing_percentage
water_ec_ms_cm_lag_4,20180,58.39
water_ph_lag_4,20180,58.39
water_flow_l_min_lag_4,20180,58.39
water_ph_lag_2,20170,58.36
water_flow_l_min_change_30m,20170,58.36
water_ec_ms_cm_change_30m,20170,58.36
water_ph_change_30m,20170,58.36
water_ec_ms_cm_lag_2,20170,58.36
water_flow_l_min_lag_2,20170,58.36
water_flow_l_min_rolling_std,20165,58.35


## Labelling Strategy

Construct Future Action Labels

The model will predict which operator action is most likely to occur within the next 60 minutes.

Labelling rules:
- Only commands occurring after the prediction timestamp are considered.
- The prediction horizon includes commands up to and including 60 minutes later.
- If multiple commands occur within the horizon, use the earliest command.
- If no command occurs within the horizon, assign `NO_ACTION`.
- Match commands to the same device to preserve device-specific behaviour.

## Action classes

In [12]:
# Define every action the model is intended to predict
ACTION_CLASSES = [
    "ACTIVATE_PUMP",
    "DEACTIVATE_PUMP",
    "ACTIVATE_FAN",
    "DEACTIVATE_FAN",
    "INSPECT_DEVICE",
    "NO_ACTION",
]

print("Target Action classes:")
for action in ACTION_CLASSES:
    print(f"- {action}")

Target Action classes:
- ACTIVATE_PUMP
- DEACTIVATE_PUMP
- ACTIVATE_FAN
- DEACTIVATE_FAN
- INSPECT_DEVICE
- NO_ACTION


## Generate future Action Labels

In [13]:
# Start with NO_ACTION
feature_df["target_action"] = "NO_ACTION"

feature_df["target_action_timestamp"] = pd.Series(
    pd.NaT,
    index=feature_df.index,
    dtype=commands_df["timestamp"].dtype,
)

horizon = pd.Timedelta(minutes=PREDICTION_HORIZON_MINUTES)

# Label each device independently
for device_id, device_features in feature_df.groupby("device_id", sort=False):
    device_commands = (
        commands_df.loc[commands_df["device_id"] == device_id]
        .sort_values("timestamp")
    )

    if device_commands.empty:
        continue

    prediction_times = device_features["timestamp"].to_numpy(
        dtype="datetime64[ns]"
    )
    command_times = device_commands["timestamp"].to_numpy(
        dtype="datetime64[ns]"
    )

    # Find the first command strictly after each prediction timestamp
    next_command_positions = np.searchsorted(
        command_times,
        prediction_times,
        side="right",
    )

    has_future_command = next_command_positions < len(device_commands)
    row_positions = np.flatnonzero(has_future_command)

    if len(row_positions) == 0:
        continue

    command_positions = next_command_positions[has_future_command]

    future_times = command_times[command_positions]
    valid_prediction_times = prediction_times[row_positions]

    # Keep future commands occurring within the prediction horizon
    within_horizon = (
        future_times - valid_prediction_times
    ) <= horizon.to_timedelta64()

    labelled_rows = row_positions[within_horizon]
    labelled_commands = command_positions[within_horizon]

    # Assign the earliest qualifying future action and its timestamp
    feature_df.loc[
        device_features.index[labelled_rows],
        "target_action",
    ] = device_commands.iloc[labelled_commands]["action"].to_numpy()

    feature_df.loc[
        device_features.index[labelled_rows],
        "target_action_timestamp",
    ] = device_commands.iloc[labelled_commands]["timestamp"].array

print("Future Action labels Generated")
print(f"Prediction Rows: {len(feature_df):,}")

Future Action labels Generated
Prediction Rows: 34,560


## Inspect the target Distribution

In [14]:
# Include all intended classes
action_distribution = (
    feature_df["target_action"]
    .value_counts()
    .reindex(ACTION_CLASSES, fill_value=0)
    .rename_axis("action")
    .reset_index(name="count")
)

action_distribution["percentage"] = (
    action_distribution["count"] / len(feature_df) * 100
).round(2)

display(action_distribution)

# Identify classes that the generated data cannot currently teach the model
missing_classes = action_distribution.loc[
    action_distribution["count"] == 0, "action"
].tolist()

if missing_classes:
    print("WARNING — classes with no labelled examples:")
    for action in missing_classes:
        print(f"- {action}")
else:
    print("All intended action classes have labelled examples.")

,action,count,percentage
0,ACTIVATE_PUMP,72,0.21
1,DEACTIVATE_PUMP,60,0.17
2,ACTIVATE_FAN,244,0.71
3,DEACTIVATE_FAN,237,0.69
4,INSPECT_DEVICE,0,0.00
5,NO_ACTION,33947,98.23


WARNING — classes with no labelled examples:
- INSPECT_DEVICE


## Validate Label Timing

In [15]:
# Verify that labelled events occur strictly after prediction time
labelled = feature_df.loc[
    feature_df["target_action"] != "NO_ACTION"
].copy()

labelled["lead_time_minutes"] = (
    labelled["target_action_timestamp"] - labelled["timestamp"]
).dt.total_seconds() / 60

assert (labelled["lead_time_minutes"] > 0).all(), (
    "A target action is not strictly in the future."
)

assert (
    labelled["lead_time_minutes"] <= PREDICTION_HORIZON_MINUTES
).all(), (
    "A target action exceeds the prediction horizon."
)

assert (
    labelled["target_action_timestamp"].notna().all()
), "A labelled action is missing its event timestamp."

assert (
    feature_df.loc[
        feature_df["target_action"] == "NO_ACTION",
        "target_action_timestamp",
    ].isna().all()
), "NO_ACTION rows unexpectedly contain action timestamps."

print("Label timing validation Passed")
print(f"Labelled future actions: {len(labelled):,}")
print(
    "Observed lead time (minutes): "
    f"{labelled['lead_time_minutes'].min():.0f}–"
    f"{labelled['lead_time_minutes'].max():.0f}"
)

Label timing validation Passed
Labelled future actions: 613
Observed lead time (minutes): 15–60


## Diagnose inspection Triggers

In [16]:
# Inspect whether the telemetry ever reaches the inspection thresholds
inspection_diagnostics = pd.DataFrame({
    "metric": ["device_health", "battery_pct"],
    "minimum": [
        telemetry_df["device_health"].min(),
        telemetry_df["battery_pct"].min(),
    ],
    "maximum": [
        telemetry_df["device_health"].max(),
        telemetry_df["battery_pct"].max(),
    ],
    "threshold": [0.75, 20.0],
    "readings_below_threshold": [
        (telemetry_df["device_health"] < 0.75).sum(),
        (telemetry_df["battery_pct"] < 20.0).sum(),
    ],
})

display(inspection_diagnostics)

# Check whether inspection commands exist in the original command history
print(
    "INSPECT_DEVICE commands generated:",
    (commands_df["action"] == "INSPECT_DEVICE").sum(),
)

,metric,minimum,maximum,threshold,readings_below_threshold
0,device_health,0.837,1.0,0.75,0
1,battery_pct,81.930,100.0,20.00,0


INSPECT_DEVICE commands generated: 0


## Record the Class Limitation

In [17]:
# Document classes that currently have no training examples
unavailable_classes = action_distribution.loc[
    action_distribution["count"] == 0, "action"
].tolist()

if unavailable_classes:
    print("Classes without training examples:")
    for action in unavailable_classes:
        print(f"- {action}")

    print(
        "The model will not learn these classes from the current Dataset"
    )
else:
    print("All intended action classes have training Examples")

Classes without training examples:
- INSPECT_DEVICE
The model will not learn these classes from the current Dataset


## Finalise the Feature Dataset

## Identify Feature Columns

In [18]:
excluded_columns = {
    "timestamp",
    "device_id",
    "target_action",
    "target_action_timestamp",
}

feature_columns = [
    column
    for column in feature_df.columns
    if column not in excluded_columns
]

X = feature_df[feature_columns].copy()
y = feature_df["target_action"].copy()

print(f"Feature rows: {len(X):,}")
print(f"Feature columns: {len(feature_columns):,}")
print(f"Target classes observed: {y.nunique()}")
print(f"Target rows: {len(y):,}")

display(X.head())

Feature rows: 34,560
Feature columns: 58
Target classes observed: 5
Target rows: 34,560


,device_category,zone,temperature_c,humidity_pct,moisture_pct,water_ph,water_ec_ms_cm,water_flow_l_min,device_health,battery_pct,...,battery_pct_rolling_mean,battery_pct_rolling_std,temperature_c_change_30m,humidity_pct_change_30m,moisture_pct_change_30m,water_ph_change_30m,water_ec_ms_cm_change_30m,water_flow_l_min_change_30m,device_health_change_30m,battery_pct_change_30m
0,Environmental,ZONE-A,23.29,71.41,49.59,NaN,NaN,NaN,0.974,100.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Environmental,ZONE-A,23.07,67.39,50.87,NaN,NaN,NaN,0.926,100.00,...,100.0000,0.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Environmental,ZONE-A,22.57,64.92,45.83,NaN,NaN,NaN,0.961,100.00,...,100.0000,0.000,-0.72,-6.49,-3.76,NaN,NaN,NaN,-0.013,0.00
3,Environmental,ZONE-A,24.00,72.27,47.24,NaN,NaN,NaN,0.973,98.87,...,99.7175,0.565,0.93,4.88,-3.63,NaN,NaN,NaN,0.047,-1.13
4,Environmental,ZONE-A,22.81,67.66,49.91,NaN,NaN,NaN,0.959,100.00,...,99.7175,0.565,0.24,2.74,4.08,NaN,NaN,NaN,-0.002,0.00


## Review Missing Values

In [19]:
# Summarise Missing values before choosing a Treatment
missing_summary = (
    X.isna()
    .sum()
    .sort_values(ascending=False)
    .rename("missing_count")
    .to_frame()
)

missing_summary["missing_percentage"] = (
    missing_summary["missing_count"] / len(X) * 100
).round(2)

display(missing_summary.loc[missing_summary["missing_count"] > 0])

,missing_count,missing_percentage
water_flow_l_min_lag_4,20180,58.39
water_ec_ms_cm_lag_4,20180,58.39
water_ph_lag_4,20180,58.39
water_flow_l_min_change_30m,20170,58.36
water_ph_change_30m,20170,58.36
water_ph_lag_2,20170,58.36
water_ec_ms_cm_lag_2,20170,58.36
water_ec_ms_cm_change_30m,20170,58.36
water_flow_l_min_lag_2,20170,58.36
water_ph_rolling_mean,20165,58.35


## Handle missing Values and Encode Categories

In [20]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# Identify categorical and numeric Features
categorical_features = X.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

numeric_features = [
    column for column in feature_columns
    if column not in categorical_features
]

# Numeric features use the median categories use the most Frequent value
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

print(f"Numeric Features: {len(numeric_features)}")
print(f"Categorical Features: {len(categorical_features)}")
print("Preprocessing Pipeline Configured")

Numeric Features: 56
Categorical Features: 2
Preprocessing Pipeline Configured


## Validate the Final Dataset 

In [21]:
# Confirm that identifiers and Future-event metadata are not Model inputs
assert "device_id" not in feature_columns
assert "timestamp" not in feature_columns
assert "target_action_timestamp" not in feature_columns

# Confirm feature and Target rows remain Aligned
assert len(X) == len(y)
assert X.index.equals(y.index)

# Verify that the intended Target is present and contains no Missing labels
assert y.notna().all()

print("Feature/target alignment Validated")
print("No target Timestamps or Device identifiers in model inputs")
print(f"Final Dataset Shape: {X.shape}")
print("\nTarget Distribution:")
display(y.value_counts().rename_axis("action").to_frame("count"))

Feature/target alignment Validated
No target Timestamps or Device identifiers in model inputs
Final Dataset Shape: (34560, 58)

Target Distribution:


,count
action,
NO_ACTION,33947
ACTIVATE_FAN,244
DEACTIVATE_FAN,237
ACTIVATE_PUMP,72
DEACTIVATE_PUMP,60


## Export

In [22]:
# Export the Engineered Dataset
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIRECTORY = PROJECT_ROOT / "data" / "generated"
DATA_DIRECTORY.mkdir(parents=True, exist_ok=True)

ENGINEERED_FEATURES_PATH = (
    DATA_DIRECTORY / "engineered_features.csv"
)

# Export Features, Target labels, and metadata
feature_df.to_csv(ENGINEERED_FEATURES_PATH, index=False)

print("Engineered Dataset Exported Successfully")
print(f"Location: {ENGINEERED_FEATURES_PATH}")
print(f"Rows: {len(feature_df):,}")
print(f"Columns: {len(feature_df.columns)}")

Engineered Dataset Exported Successfully
Location: /Users/quendrew/Desktop/Smart-X-ML/data/generated/engineered_features.csv
Rows: 34,560
Columns: 62
